# **Changing Election Outcomes: Comparing Electoral College Methods To The Popular Vote**

### This project investigates how accurately methods of awarding Electoral College Votes hold to the principles of representational government. Three methods of awarding Electoral College Votes are tested.

1. **Standard Winner Take All** - A state's highest popular vote candidate is awarded all of the state's Electoral College Votes

1. **Full Divide** - A state's Electoral College Votes are awarded by proportionally by the number of popular votes for each candidate

1. **Nebraska-like** – A state's Representative Electoral College Votes are awarded by the proportionally by popular votes for each candidate. However, a state's Senatorial Electoral College Votes are awarded to the highest popular vote candidate


### These three methods are tested on Electoral Colleges based on the current "Capped" number of representatives and an "Uncapped" number of representatives where each state is awarded as many representatives possible under the limit of one representative for every 30,000 people. Accuracy of representation is determined by how closely Electoral College Vote percentages are to the national popular vote percentages. In order to gauge the precision of these methods, each is tested using the state voting numbers from the U.S. Presidential Elections in 2008, 2012, 2016, and 2020.

##### For more information on the history of these decisions and the details of the apportionment methods, please visit the powerpoint: PP_Electoral_System_Explanation.pptx

## **Cleaning Obtained Data for Analysis**

### The needed data for the elections 2008-2020 was isolated from larger datasets obtained from the Federal Election Comission and the U.S. Census Bureau through Excel. This data was further refined into SQL tables within the *Cleaning_Process* database with the following column format
* FEC_ID
* Candidate_Name
* Candidate_Votes
* Apportionment_pop
* Num_Reps_Capped
* State_Vote_Total

##### For more information on the data sources and the isolated data, please visit the *Raw_Data* subfolder of the *Data* folder.
##### For more information on the data cleaning process and resulting database, please visit the *Cleaning_Process_and_Cleaned_Data* subfolder of the *Data* folder.

## **Explaination of analysis**

#### Preliminary imports, database assignments, and ensuring data type accuracy.

In [ ]:
import pandas as pd
import sqlite3


dbfile = input("Enter database retrieving table from: ")
dbfile2 = input("Enter a name to save your new database as: ") + ".db"
table_name = input("Enter table name to search: ")

con = sqlite3.connect(dbfile)


DF = pd.read_sql_query(f'select * from {table_name}', con)

DF[["Apportionment_pop", "Candidate_Votes", "State_Vote_Total"]] = DF[["Apportionment_pop", "Candidate_Votes", "State_Vote_Total"]].apply(pd.to_numeric)

### This section awards one Electoral Representative Vote, equal to the least populous state, to Washington D.C. within the current Capped Electoral College system as Washington D.C.'s apportionment data is not provided. For the Uncapped Electoral College System, the state's Electoral College Votes are awarded based on the quotient of the apportionment population divided by 30000. Once again Washington D.C. is awarded the same number of representative Electoral College Votes as the least populous state.

In [ ]:
# Electoral College Capped
# Assigns 1 Electoral Vote for Washington D.C. null values

DF["Num_Reps_Capped"] = DF["Num_Reps_Capped"].fillna(1)

# Uncapped Electoral College
# Uses quotient of apportionment population division to assign Rep. Electoral Votes
# Assigns the lowest number of Electoral Votes for a state to Washington D.C. null values

DF = DF.assign(Num_Reps_Uncapped=DF["Apportionment_pop"] // 30000)

DF["Num_Reps_Uncapped"] = DF["Num_Reps_Uncapped"].fillna(DF["Num_Reps_Uncapped"].min())


### For the Capped Electoral College and Uncapped Electoral College, the number of popular votes needed for one Electoral Vote, the number of Electoral Votes awarded to each candidate based on their popular vote totals, and the remainder of popular votes for each candidate are calculated with code below. Please note how the full divide methods which divides the whole amount of state Electoral Votes adds two to the number of Representative Electoral Votes to compensate for the Senatorial Electoral Votes which the other methods add on based on the plurality winner of the popular vote.

In [ ]:
# Capped EC NE-Like Style
# Calculates the number of popular votes needed for an Electoral Vote,
# the number of Electoral Votes for each candidate, 
# and the remainder of popular votes for each candidate

DF = DF.assign(CapEC_NE_VotePop=DF["State_Vote_Total"] / DF["Num_Reps_Capped"])
DF = DF.assign(CapEC_NE_CanVotes=DF["Candidate_Votes"] // DF["CapEC_NE_VotePop"])
DF = DF.assign(CapEC_NE_CanRemainder=DF["Candidate_Votes"] % DF["CapEC_NE_VotePop"])

# Capped EC Full Divide
# Calculates the number of popular votes needed for an Electoral Vote,
# the number of Electoral Votes for each candidate, 
# and the remainder of popular votes for each candidate

DF = DF.assign(CapEC_Full_VotePop=DF["State_Vote_Total"] / (DF["Num_Reps_Capped"] + 2))
DF = DF.assign(CapEC_Full_CanVotes=DF["Candidate_Votes"] // DF["CapEC_Full_VotePop"])
DF = DF.assign(CapEC_Full_CanRemainder=DF["Candidate_Votes"] % DF["CapEC_Full_VotePop"])

# Uncapped EC NE-Like style
# Calculates the number of popular votes needed for an Electoral Vote,
# the number of Electoral Votes for each candidate, 
# and the remainder of popular votes for each candidate

DF = DF.assign(UnCapEC_NE_VotePop=DF["State_Vote_Total"] / DF["Num_Reps_Uncapped"])
DF = DF.assign(UnCapEC_NE_CanVotes=DF["Candidate_Votes"] // DF["UnCapEC_NE_VotePop"])
DF = DF.assign(UnCapEC_NE_CanRemainder=DF["Candidate_Votes"] % DF["UnCapEC_NE_VotePop"])

# Uncapped EC full divide
# Calculates the number of popular votes needed for an Electoral Vote,
# the number of Electoral Votes for each candidate, 
# and the remainder of popular votes for each candidate

DF = DF.assign(UnCapEC_Full_VotePop=DF["State_Vote_Total"] / (DF["Num_Reps_Uncapped"] + 2))
DF = DF.assign(UnCapEC_Full_CanVotes=DF["Candidate_Votes"] // DF["UnCapEC_Full_VotePop"])
DF = DF.assign(UnCapEC_Full_CanRemainder=DF["Candidate_Votes"] % DF["UnCapEC_Full_VotePop"])

con2 = sqlite3.connect(dbfile2)
cursor = con2.cursor()

DF.to_sql('LG_Elect', con2, if_exists='replace', index=False)
con.close()

### In the NE-Like and the Full Divide methods for the Capped and Uncapped Electoral Colleges, the remaining Electoral Votes for each state are determined. The candidates in each method are separated by state, ranked by remaining popular vote, and sorted in descending order. The candidates whose rank is lower or equal to the remaining Electoral Votes for their state per method are awarded an additional Electoral College Vote added to the candidates' total Electoral College Votes.

``` SQL
-- Assigns Electoral Votes based on popular vote remainders

CREATE TABLE RemVotes_Assigned AS

/* 
Determines how many Electoral Votes (Representative Electoral Votes for NE-Like Style and all for Full Divide)
are still to be awarded by remainder for each method and state after intial Electoral Votes were awarded. 
This is accomplished by subtracting the awarded Electoral Votes from the apportioned Electoral Votes for each state per method. 
*/

WITH Remainder AS (
SELECT STATE, Num_Reps_Capped - SUM(CapEC_NE_CanVotes)as "CapEC_NE_CanRemVote", 
(Num_Reps_Capped + 2) - SUM(CapEC_Full_CanVotes) as "CapEC_Full_CanRemVote",
Num_Reps_Uncapped - SUM(UnCapEC_NE_CanVotes) as "UnCapEC_NE_CanRemVote", 
(Num_Reps_Uncapped + 2) - SUM(UnCapEC_Full_CanVotes) as "UnCapEC_Full_CanRemVote"
FROM LG_Elect
GROUP BY STATE
),

-- Joins number of remaining Electoral Votes to main table by method and state

LG_Remain AS (
SELECT LG_Elect.*, Remainder.CapEC_NE_CanRemVote, Remainder.CapEC_Full_CanRemVote,
Remainder.UnCapEC_NE_CanRemVote, Remainder.UnCapEC_Full_CanRemVote
FROM LG_Elect
LEFT JOIN Remainder
ON LG_Elect.State = Remainder.State
),

-- For all methods, ranks candidates in each state by remaining popular votes with 1 being the highest

Ranking AS (
SELECT *,
CAST(ROW_NUMBER() OVER (PARTITION BY STATE ORDER BY CapEC_NE_CanRemainder DESC, Candidate_Votes DESC) AS REAL) AS CapEC_NE_CanRemain_Rank,
CAST(ROW_NUMBER() OVER (PARTITION BY STATE ORDER BY CapEC_Full_CanRemainder DESC, Candidate_Votes DESC) AS REAL) AS CapEC_Full_CanRemain_Rank,
CAST(ROW_NUMBER() OVER (PARTITION BY STATE ORDER BY UnCapEC_NE_CanRemainder DESC, Candidate_Votes DESC) AS REAL) AS UnCapEC_NE_CanRemain_Rank,
CAST(ROW_NUMBER() OVER (PARTITION BY STATE ORDER BY UnCapEC_Full_CanRemainder DESC, Candidate_Votes DESC) AS REAL) AS UnCapEC_Full_CanRemain_Rank
FROM LG_Remain
)

/*
For each method, apportions remaining Electoral Votes of each state (one for each candidate) 
to the highest ranking candidates by popular vote remainder as new table RemVotes_Assigned
*/

SELECT
STATE,
Candidate_Name,
CASE
WHEN Ranking.CapEC_NE_CanRemain_Rank <= Ranking.CapEC_NE_CanRemVote
THEN 1 ELSE 0 END AS CapEC_NE_RemainAdd, 
CASE
WHEN Ranking.CapEC_Full_CanRemain_Rank <= Ranking.CapEC_Full_CanRemVote
THEN 1 ELSE 0 END AS CapEC_Full_RemainAdd, 
CASE
WHEN Ranking.UnCapEC_NE_CanRemain_Rank <= Ranking.UnCapEC_NE_CanRemVote
THEN 1 ELSE 0 END AS UnCapEC_NE_RemainAdd,
CASE
WHEN Ranking.UnCapEC_Full_CanRemain_Rank <= Ranking.UnCapEC_Full_CanRemVote
THEN 1 ELSE 0 END AS UnCapEC_Full_RemainAdd
FROM Ranking;




-- For each method, adds remainder Electoral Votes to previous candidate Electoral Vote totals by state as a new table

CREATE TABLE Rep_Vote_w_RemAdd
AS
Select LG_Elect.STATE, LG_Elect.FEC_ID, LG_Elect.Candidate_Name, LG_Elect.Candidate_Votes, LG_Elect.Num_Reps_Capped,
LG_Elect.Num_Reps_Uncapped,
CAST((LG_Elect.CapEC_NE_CanVotes + RemVotes_Assigned.CapEC_NE_RemainAdd) AS REAL) AS CapEC_NE_CanVote_ARem,
CAST((LG_Elect.CapEC_Full_CanVotes + RemVotes_Assigned.CapEC_Full_RemainAdd) AS REAL) AS CapEC_Full_CanVote_ARem, 
CAST((LG_Elect.UnCapEC_NE_CanVotes + RemVotes_Assigned.UnCapEC_NE_RemainAdd) AS REAL) AS UnCapEC_NE_CanVote_ARem, 
CAST((LG_Elect.UnCapEC_Full_CanVotes + RemVotes_Assigned.UnCapEC_Full_RemainAdd) AS REAL) AS UnCapEC_Full_CanVote_ARem
FROM LG_Elect
LEFT JOIN RemVotes_Assigned
ON LG_Elect.STATE = RemVotes_Assigned.STATE
AND LG_Elect.Candidate_Name = RemVotes_Assigned.Candidate_Name;

```

### Candidates are ranked by popular vote to determine the winners for the standard Electoral College and extra Senatorial Electoral College Votes for the NE-Like Style, while awarding zero Electoral Votes to the non-first candidates. These results are added to the total Electoral College Votes for each candidate by state and method. Finally a new table is created to present clear results without any extraneous data or columns.

```SQL

-- Plurality Electoral Vote Assignment


CREATE TABLE Sen_ECvote_assigned AS

-- Ranks states' candidates by popular vote and initializes columns CapEC_Standard, UnCapEC_Standard, and SenNum with values of zero

SELECT *,
CAST(ROW_NUMBER() OVER (PARTITION BY STATE ORDER BY Candidate_Votes DESC) AS REAL) AS Sen_Rank,
CAST(0 AS REAL) AS CapEC_Standard,
CAST(0 AS REAL) AS UnCapEC_Standard,
CAST(0 AS REAL) AS SenNum
FROM Rep_Vote_w_RemAdd;

/* 
Gives state popular vote winner all state Electoral College Votes in Standard Electoral College method 
and two additional Senatorial Electoral College Votes for the NE Style Method
*/

UPDATE Sen_ECvote_assigned
SET
CapEC_Standard = Num_Reps_Capped + 2,
UnCapEC_Standard = Num_Reps_Uncapped + 2,
SenNum = 2
WHERE Sen_Rank = 1;


-- Totals Electoral College Votes for each methods and removes unneeded data for final comparisions with national popular vote

CREATE TABLE Total
AS
SELECT STATE, FEC_ID, Candidate_Name, Candidate_Votes, CAST((Num_Reps_Capped + 2) AS REAL) AS Capped_Num_State_EC_Votes,
CAST((Num_Reps_Uncapped + 2) AS REAL) AS UnCapped_Num_State_EC_Votes,
CapEC_Standard AS Capped_EC_Standard,
CAST((CapEC_NE_CanVote_ARem + SenNum) AS REAL) AS Capped_EC_NE_Candidate_Vote, 
CapEC_Full_CanVote_ARem AS Capped_EC_Full_Candidate_Vote, 
UnCapEC_Standard AS UnCapped_EC_Standard,
CAST((UnCapEC_NE_CanVote_ARem + SenNum) AS REAL) AS UnCapped_EC_NE_Candidate_Vote,
UnCapEC_Full_CanVote_ARem AS UnCapped_EC_Full_Candidate_Vote
FROM Sen_ECvote_assigned;


```

##### For full computational code to create SQL databases and tables for this analysis project, please visit the *08_20_DB_construction_code.py* file in the *Analysis_Code* folder. 

##### For full computational code to create SQL databases and tables for other election years, please visit the *General_analysis_DB_code.py* file in the *Analysis_Code* folder.

## **Results and Conclusions**

### The data on the Presidential Elections of 2008, 2012, 2016, and 2020 reveals five conclusions when comparing the national popular vote to the Standard Winner Take All, Full Divide, and Nebraska-like methods with the Capped or Uncapped Electoral College:

1. The percentages of the Winner Take All methods seem to inflate the Electoral College percentages of leading candidates and suppress the Electoral College percentages of lagging competitors making their results significantly different than the Nebraska-like, Full Divide, and the popular vote percentages.

1. Due to the inflation and suppression of the Winner Take All method, utilized by most states currently, the 2016 U.S. Presidential Election had a decisive winner. If the Nebraska-like method, Full Divide method, or popular vote had been used instead, no candidate would have secured over 50 percent of the Electoral College and the 2016 Presidential Election would have been decided by Congressional vote.

1. Methods besides the Winner Take All method allows for third party candidates to receive Electoral College Votes.

1. Methods using the Uncapped Electoral College are more closely aligned with the Popular Vote percentages than those utilizing the standard Capped Electoral College.

1. The percentages of Uncapped Full Divide and Uncapped Nebraska-like methods stay within a fifth of a percent of the popular vote percentages. As each method has a smaller difference in two of the elections, one method cannot be determined to be superior without further analysis and increased sample sizes.

##### For visual comparisons and a presentation of these findings, please visit the **Visualizations** folder.